# Multimodal Emotion Recognition — Held-out Evaluation

Bu notebook RAVDESS + CREMA-D **speaker-independent test split** üzerinde üç koşulu değerlendirir:

1. **Vision-only**: her klipten 8 kare örnekle, vision modeline ver, ortalamayı al
2. **SER-only**: klibin tüm ses kısmını SER modeline ver
3. **Vision + SER fusion**: `0.588 × vision + 0.412 × ser` (live sistemdeki ağırlıklardan renormalize)

NLP bu evaluation'da değil — RAVDESS/CREMA-D cümleleri emotion-neutral text içeriyor.

**Çıktılar**: 3 confusion matrix, classification report, karşılaştırma tablosu, hepsi `Drive/BitirmeProjesi/multimodal_eval/` altında.

**Çalıştırma**: hücreleri sırayla çalıştır. Bir hücre patlarsa o hücreyi tekrar çalıştır (çoğu durumda bu işi çözer). Çözmezse hata mesajını bana yapıştır.

## 1. Drive mount + GPU kontrol

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}')
print(f'GPU: {gpus if gpus else "YOK — CPU üzerinde çalışacak, biraz yavaş olur ama çalışır"}')

## 2. Sabitler ve yollar

Drive yapın şöyle olmalı (screenshot'tan görünen):
```
BitirmeProjesi/
├── ser_cnn_bilstm/
│   ├── ser_cnn_bilstm.keras
│   ├── ser_cnn_bilstm_best.keras
│   ├── ser_cnn_bilstm_meta.json
│   ├── ser_cnn_bilstm_report.txt
│   └── waveforms_cache.npz
└── vision_ferplus/
    └── vision_ferplus.keras  ← bunun varlığını kontrol edeceğiz
```

Vision modelin başka klasördeyse aşağıdaki `VISION_MODEL` yolunu güncelle.

In [ ]:
import os, json
import numpy as np

DRIVE_ROOT  = '/content/drive/MyDrive/BitirmeProjesi'
SER_DIR     = f'{DRIVE_ROOT}/ser_cnn_bilstm'
VISION_DIR  = f'{DRIVE_ROOT}/vision_ferplus'

SER_MODEL   = f'{SER_DIR}/ser_cnn_bilstm.keras'
SER_META    = f'{SER_DIR}/ser_cnn_bilstm_meta.json'
SER_CACHE   = f'{SER_DIR}/waveforms_cache.npz'
VISION_MODEL = f'{VISION_DIR}/vision_ferplus.keras'

WORK_DIR    = '/content/eval_work'
RAVDESS_DIR = f'{WORK_DIR}/ravdess'
CREMAD_DIR  = f'{WORK_DIR}/crema_d'
OUT_DIR     = f'{DRIVE_ROOT}/multimodal_eval'

for d in [WORK_DIR, RAVDESS_DIR, CREMAD_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

# 6 sınıf (alfabetik, training ile aynı)
CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
NUM_CLASSES = 6

# Fusion ağırlıkları — live sistem (0.50, 0.35, 0.20) → NLP=0 → renormalize
W_VISION = 0.50 / (0.50 + 0.35)
W_SER    = 0.35 / (0.50 + 0.35)
print(f'Fusion: vision={W_VISION:.3f}, SER={W_SER:.3f}, NLP=0')

# Audio params (TRAINING ile birebir aynı olmalı)
SR              = 16000
TARGET_DURATION = 3.0
TARGET_LEN      = int(SR * TARGET_DURATION)
N_MELS          = 128
N_FFT           = 1024
HOP_LENGTH      = 256
TRIM_TOP_DB     = 30

# Vision params
VISION_IMG_SIZE = 48
FRAMES_PER_CLIP = 8

# Dosya kontrolü
for name, p in [('vision', VISION_MODEL), ('SER', SER_MODEL), ('meta', SER_META), ('cache', SER_CACHE)]:
    if os.path.exists(p):
        print(f'  ✓ {name}: {p} ({os.path.getsize(p)/1e6:.1f} MB)')
    else:
        print(f'  ✗ {name}: BULUNAMADI → {p}')

## 3. Modelleri yükle

In [ ]:
print('Loading vision model...')
vision_model = tf.keras.models.load_model(VISION_MODEL)
print(f'  Vision: {vision_model.count_params():,} params, input {vision_model.input_shape}')

print('Loading SER model...')
ser_model = tf.keras.models.load_model(SER_MODEL)
print(f'  SER: {ser_model.count_params():,} params, input {ser_model.input_shape}')

import cv2
HAAR = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(HAAR)
print(f'Face cascade loaded.')

## 4. Test split'i bul

`meta.json` ve `waveforms_cache.npz` içeriğini ekrana basıyoruz. Test dosya listesi bu iki kaynaktan birinde olmalı. Yoksa hata verir ve **çıktıyı bana yapıştırırsın**, ona göre adapte ederim.

In [ ]:
# meta.json içeriği
with open(SER_META, 'r') as f:
    meta = json.load(f)

print('=== meta.json keys & types ===')
for k, v in meta.items():
    if isinstance(v, list):
        print(f'  {k}: list[{len(v)}]', f'-> ilk: {v[0] if v else "(boş)"}' if len(v) else '')
    elif isinstance(v, dict):
        print(f'  {k}: dict, keys={list(v.keys())}')
    else:
        print(f'  {k}: {v}')

# waveforms_cache.npz içeriği
print('\n=== waveforms_cache.npz keys & shapes ===')
cache = np.load(SER_CACHE, allow_pickle=True)
for k in cache.files:
    arr = cache[k]
    if arr.dtype == object or arr.dtype.kind == 'U':
        print(f'  {k}: shape={arr.shape}, dtype={arr.dtype} | sample: {arr.flat[0] if arr.size else "(boş)"}')
    else:
        print(f'  {k}: shape={arr.shape}, dtype={arr.dtype}')

In [ ]:
# Yaygın isimleri dene; test_files ve test_labels'ı bul
test_files  = None
test_labels = None

# Meta.json üzerinden
for key_pair in [('test_files','test_labels'), ('test_paths','test_labels'),
                 ('X_test_files','y_test'), ('test_filenames','test_y')]:
    if key_pair[0] in meta and key_pair[1] in meta:
        test_files  = meta[key_pair[0]]
        test_labels = meta[key_pair[1]]
        print(f'meta.json bulundu: {key_pair}')
        break

# Cache üzerinden (alternatif)
if test_files is None:
    for key_pair in [('test_files','test_labels'), ('X_test_files','y_test'),
                     ('test_filenames','test_y')]:
        if key_pair[0] in cache.files and key_pair[1] in cache.files:
            test_files  = cache[key_pair[0]].tolist()
            test_labels = cache[key_pair[1]].tolist()
            print(f'cache bulundu: {key_pair}')
            break

if test_files is None:
    print('!!! Test split BULUNAMADI.')
    print('!!! Üstteki iki çıktıyı (meta keys + cache keys) bana yapıştır.')
    raise SystemExit('Need test split info')

test_files = list(map(str, test_files))
test_labels = [int(x) for x in test_labels]
print(f'\nTest set: {len(test_files)} klip, {len(set(test_labels))} sınıf')
print(f'İlk dosya örneği: {test_files[0]}')

In [ ]:
# Class dağılımı + source dağılımı
import pandas as pd
from collections import Counter

def detect_source(fname):
    """RAVDESS: 03-01-04-01-02-01-12.wav (7 alan, dash ile)
       CREMA-D: 1001_DFA_ANG_XX.wav (4 alan, underscore ile)"""
    base = os.path.basename(fname).rsplit('.', 1)[0]
    if '-' in base and len(base.split('-')) == 7:
        return 'ravdess'
    if '_' in base and len(base.split('_')) == 4:
        return 'crema_d'
    return 'unknown'

test_df = pd.DataFrame({
    'filename': [os.path.basename(f) for f in test_files],
    'orig_path': test_files,
    'label_idx': test_labels,
    'label': [CLASSES[i] for i in test_labels],
    'source': [detect_source(f) for f in test_files],
})

print('Sınıf dağılımı:')
print(test_df['label'].value_counts())
print('\nKaynak dağılımı:')
print(test_df['source'].value_counts())
print('\nSource × Class:')
print(pd.crosstab(test_df['source'], test_df['label']))

## 5. RAVDESS video indir (sadece test'teki actor'ler)

RAVDESS filename'in 7. alanı (`actor`) = aktör ID (01-24). Test setinde hangi aktörler varsa sadece o aktörlerin video zip'lerini indireceğiz.

In [ ]:
import urllib.request
from tqdm import tqdm

ravdess_files = test_df[test_df['source']=='ravdess']['filename'].tolist()

if not ravdess_files:
    print('Test setinde RAVDESS yok, atlanıyor.')
else:
    # Aktör ID'leri
    ravdess_actors = sorted({int(f.split('-')[-1].split('.')[0]) for f in ravdess_files})
    print(f'Test setinde RAVDESS aktörleri: {ravdess_actors}')
    print(f'Bunun için {len(ravdess_actors)} adet aktör zip\'i indirilecek (~700MB her biri)')
    
    base_url = 'https://zenodo.org/record/1188976/files'
    
    for actor in ravdess_actors:
        zname = f'Video_Speech_Actor_{actor:02d}.zip'
        url   = f'{base_url}/{zname}'
        local = f'{RAVDESS_DIR}/{zname}'
        target = f'{RAVDESS_DIR}/Actor_{actor:02d}'
        
        if os.path.isdir(target):
            print(f'  {actor:02d}: zaten unzipped, atlandı')
            continue
        
        if not os.path.exists(local):
            print(f'  {actor:02d}: indiriliyor → {url}')
            try:
                urllib.request.urlretrieve(url, local)
            except Exception as e:
                print(f'    HATA: {e}')
                continue
        
        print(f'  {actor:02d}: unzipping...')
        os.system(f'cd "{RAVDESS_DIR}" && unzip -q -o "{zname}"')
        os.remove(local)  # zip'i sil, yer aç
    
    # Tüm .mp4 dosyalarını listele
    import glob
    mp4s = glob.glob(f'{RAVDESS_DIR}/**/*.mp4', recursive=True)
    print(f'\nToplam {len(mp4s)} RAVDESS .mp4 indirildi')

## 6. CREMA-D video indir

CREMA-D videoları GitHub repo'sunda `VideoFlash/` altında `.flv` formatında. Sadece test setindeki dosyaları tek tek indireceğiz (~700 küçük dosya, ~5-10 dk).

In [ ]:
crema_files = test_df[test_df['source']=='crema_d']['filename'].tolist()

if not crema_files:
    print('Test setinde CREMA-D yok, atlanıyor.')
else:
    print(f'Test setinde {len(crema_files)} CREMA-D dosyası var')
    
    flash_dir = f'{CREMAD_DIR}/VideoFlash'
    os.makedirs(flash_dir, exist_ok=True)
    
    base_url = 'https://github.com/CheyneyComputerScience/CREMA-D/raw/master/VideoFlash'
    
    failed = []
    for fname in tqdm(crema_files, desc='CREMA-D video'):
        base = os.path.basename(fname).rsplit('.',1)[0]
        flv_name = f'{base}.flv'
        local = f'{flash_dir}/{flv_name}'
        
        if os.path.exists(local) and os.path.getsize(local) > 1000:
            continue
        
        url = f'{base_url}/{flv_name}'
        try:
            urllib.request.urlretrieve(url, local)
        except Exception as e:
            failed.append((flv_name, str(e)))
    
    print(f'\nBaşarılı: {len(crema_files) - len(failed)}/{len(crema_files)}')
    if failed:
        print(f'\nBaşarısız ({len(failed)}):')
        for fn, err in failed[:10]:
            print(f'  {fn}: {err}')

## 7. Test setini yerel dosyalarla eşle

In [ ]:
import glob

# RAVDESS .mp4 dosyalarını filename → path olarak indexle
ravdess_index = {}
for p in glob.glob(f'{RAVDESS_DIR}/**/*.mp4', recursive=True):
    # RAVDESS video filename: 02-01-04-01-02-01-12.mp4 (audio'da 03 yerine 02)
    # Audio: 03-01-04-...   Video: 02-01-04-...
    # 2. alan: 01=speech
    base = os.path.basename(p).rsplit('.',1)[0]
    ravdess_index[base] = p

crema_index = {}
for p in glob.glob(f'{CREMAD_DIR}/VideoFlash/*.flv'):
    base = os.path.basename(p).rsplit('.',1)[0]
    crema_index[base] = p

print(f'RAVDESS yerel: {len(ravdess_index)} dosya')
print(f'CREMA-D yerel: {len(crema_index)} dosya')

def resolve_video(test_filename, source):
    """Test setindeki audio filename'i video filename'e map et ve path döndür."""
    base = os.path.basename(test_filename).rsplit('.',1)[0]
    if source == 'ravdess':
        # Audio: 03-01-...  Video: 02-01-...
        parts = base.split('-')
        if parts[0] == '03':
            parts[0] = '02'
        video_base = '-'.join(parts)
        return ravdess_index.get(video_base)
    elif source == 'crema_d':
        return crema_index.get(base)
    return None

test_df['video_path'] = test_df.apply(
    lambda r: resolve_video(r['filename'], r['source']), axis=1
)
found = test_df['video_path'].notna().sum()
missing = test_df['video_path'].isna().sum()
print(f'\nEşleştirme: {found} bulundu, {missing} eksik')
if missing > 0:
    print('İlk 5 eksik:')
    print(test_df[test_df['video_path'].isna()][['filename','source']].head())

## 8. Vision tahmin fonksiyonu + tüm test üzerinde çalıştır

In [ ]:
def predict_vision_for_video(video_path, n_frames=FRAMES_PER_CLIP):
    """Videodan n_frames kare örnekle, her birinde yüz tespit edip vision tahmin yap,
       ortalamayı döndür. Tahmin yapılamazsa None."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None
    
    # n_frames+1 eşit aralıklı, başını ve sonunu atla
    indices = [int(total * (i+1) / (n_frames+1)) for i in range(n_frames)]
    
    faces_batch = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40,40))
        
        if len(faces) > 0:
            x,y,w,h = max(faces, key=lambda b: b[2]*b[3])  # en büyük yüz
            face = gray[y:y+h, x:x+w]
        else:
            # Yüz tespit edilemediyse merkez crop
            H, W = gray.shape
            s = min(H, W) // 2
            face = gray[H//2-s//2:H//2+s//2, W//2-s//2:W//2+s//2]
        
        if face.size == 0:
            continue
        face = cv2.resize(face, (VISION_IMG_SIZE, VISION_IMG_SIZE))
        faces_batch.append(face[..., None].astype(np.float32))
    
    cap.release()
    
    if not faces_batch:
        return None
    
    batch = np.stack(faces_batch)
    probs = vision_model.predict(batch, verbose=0)
    return probs.mean(axis=0)

# Test
sample = test_df[test_df['video_path'].notna()].iloc[0]
out = predict_vision_for_video(sample['video_path'])
print(f'Test örnek ({sample["filename"]}, label={sample["label"]}):')
print(f'  Vision probs: {out}')
if out is not None:
    print(f'  Top: {CLASSES[int(np.argmax(out))]} ({out.max():.3f})')

In [ ]:
# Tüm test üzerinde vision predictions
vision_probs_all = []
vision_failed = 0

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Vision'):
    if row['video_path'] is None or not os.path.exists(row['video_path']):
        vision_probs_all.append(None)
        vision_failed += 1
        continue
    probs = predict_vision_for_video(row['video_path'])
    if probs is None:
        vision_failed += 1
    vision_probs_all.append(probs)

print(f'\nVision tamamlandı. Başarılı: {len(test_df) - vision_failed}, Başarısız: {vision_failed}')

## 9. SER tahmin (cache'ten direkt)

Cache'te `X_test` veya benzer bir key varsa, training-time test setinin preprocessed waveformları zaten orada. Direkt model'e veriyoruz. Yoksa raw audio dosyalarından üreteceğiz.

In [ ]:
# Cache'te test waveformları var mı?
ser_probs_all = None
for key in ['X_test', 'test_waveforms', 'test_features', 'X_test_logmel']:
    if key in cache.files:
        X_test = cache[key]
        print(f'Cache key bulundu: {key} shape={X_test.shape}')
        # Log-mel mi yoksa raw waveform mu?
        if X_test.ndim == 4:  # log-mel: (N, mels, time, 1)
            print('Log-mel features bulundu, direkt model.predict')
            ser_probs_all = ser_model.predict(X_test, batch_size=32, verbose=1)
        elif X_test.ndim == 2:  # waveforms: (N, samples)
            print('Raw waveform bulundu, log-mel hesaplayıp model.predict')
            import librosa
            mels = []
            for y in tqdm(X_test, desc='log-mel'):
                mel = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                                      hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0)
                lm = librosa.power_to_db(mel, ref=np.max)
                lm = (lm - lm.mean()) / (lm.std() + 1e-6)
                mels.append(lm.astype(np.float32))
            X_logmel = np.stack(mels)[..., None]
            ser_probs_all = ser_model.predict(X_logmel, batch_size=32, verbose=1)
        break

if ser_probs_all is None:
    print('Cache\'te test waveformları bulunamadı, audio dosyalarından üretiyorum...')
    # Audio'yu video'dan çıkar (RAVDESS .mp4 ve CREMA-D .flv'de ses var)
    import librosa
    
    def predict_ser_from_video(video_path):
        y, sr = librosa.load(video_path, sr=SR, mono=True)
        y_trim, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
        if len(y_trim) < SR * 0.3:
            return None
        # Fix length
        if len(y_trim) > TARGET_LEN:
            start = (len(y_trim) - TARGET_LEN) // 2
            y_fixed = y_trim[start:start+TARGET_LEN]
        else:
            y_fixed = np.pad(y_trim, (0, TARGET_LEN - len(y_trim)))
        mel = librosa.feature.melspectrogram(y=y_fixed, sr=SR, n_fft=N_FFT,
                                              hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0)
        lm = librosa.power_to_db(mel, ref=np.max)
        lm = (lm - lm.mean()) / (lm.std() + 1e-6)
        return ser_model.predict(lm[None,...,None], verbose=0)[0]
    
    ser_probs_all = []
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='SER'):
        if row['video_path'] is None or not os.path.exists(row['video_path']):
            ser_probs_all.append(None)
            continue
        ser_probs_all.append(predict_ser_from_video(row['video_path']))
    ser_probs_all = np.array([p if p is not None else np.zeros(NUM_CLASSES) for p in ser_probs_all])

print(f'\nSER probs shape: {np.array(ser_probs_all).shape}')

## 10. Fusion + metrikler

In [ ]:
# Sadece her iki modalitenin de başarılı olduğu örnekleri tutalım (fair karşılaştırma)
valid_mask = np.array([v is not None for v in vision_probs_all]) & \
             np.array([s is not None and (np.array(s).sum() > 0) for s in ser_probs_all])

print(f'Hem vision hem SER başarılı: {valid_mask.sum()}/{len(test_df)} klip')

y_true = np.array(test_df['label_idx'])[valid_mask]
vis_probs = np.stack([v for v,m in zip(vision_probs_all, valid_mask) if m])
ser_probs = np.stack([np.array(s) for s,m in zip(ser_probs_all, valid_mask) if m])
fus_probs = W_VISION * vis_probs + W_SER * ser_probs

y_pred_vis = vis_probs.argmax(axis=1)
y_pred_ser = ser_probs.argmax(axis=1)
y_pred_fus = fus_probs.argmax(axis=1)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results = {}
for name, y_pred in [('Vision-only', y_pred_vis),
                     ('SER-only',    y_pred_ser),
                     ('Fusion',      y_pred_fus)]:
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=CLASSES,
                                    labels=list(range(NUM_CLASSES)), digits=3, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    results[name] = {'accuracy': float(acc), 'report': report, 'cm': cm.tolist()}
    print(f'\n============ {name} (acc={acc:.4f}) ============')
    print(report)

## 11. Confusion matrix grafikleri

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (name, res) in zip(axes, results.items()):
    cm = np.array(res['cm'])
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                cbar=False, vmin=0, vmax=1)
    ax.set_title(f'{name} (acc={res["accuracy"]:.3f})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {OUT_DIR}/confusion_matrices.png')

## 12. Karşılaştırma tablosu + Drive'a kaydet

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

rows = []
for name, y_pred in [('Vision-only', y_pred_vis),
                     ('SER-only',    y_pred_ser),
                     ('Fusion',      y_pred_fus)]:
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred,
                                                  labels=list(range(NUM_CLASSES)),
                                                  average='macro', zero_division=0)
    rows.append({'Modality': name, 'Accuracy': acc, 'Macro Precision': p,
                 'Macro Recall': r, 'Macro F1': f})

summary = pd.DataFrame(rows).set_index('Modality')
print(summary.to_string(float_format='%.4f'))
summary.to_csv(f'{OUT_DIR}/summary.csv', float_format='%.4f')

# Klip bazında tahmin
pred_df = test_df[valid_mask].copy().reset_index(drop=True)
pred_df['vision_pred'] = [CLASSES[i] for i in y_pred_vis]
pred_df['ser_pred']    = [CLASSES[i] for i in y_pred_ser]
pred_df['fusion_pred'] = [CLASSES[i] for i in y_pred_fus]
pred_df['vision_conf'] = vis_probs.max(axis=1)
pred_df['ser_conf']    = ser_probs.max(axis=1)
pred_df['fusion_conf'] = fus_probs.max(axis=1)
pred_df.to_csv(f'{OUT_DIR}/per_clip_predictions.csv', index=False)

# JSON özet
with open(f'{OUT_DIR}/results.json', 'w') as f:
    json.dump({
        'num_test_clips_total': len(test_df),
        'num_test_clips_valid': int(valid_mask.sum()),
        'classes': CLASSES,
        'fusion_weights': {'vision': W_VISION, 'ser': W_SER, 'nlp': 0.0},
        'results': {k: {'accuracy': v['accuracy'], 'cm': v['cm']} for k,v in results.items()},
    }, f, indent=2)

print(f'\nTüm çıktılar kaydedildi: {OUT_DIR}/')
for f in os.listdir(OUT_DIR):
    print(f'  {f}')